In [ ]:
###
# Cell 01
###

import torch
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
from google.colab import files

from utils.network import SimpleCNN

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
###
# Cell 02
###

model = SimpleCNN()

model_path = "/content/drive/MyDrive/cnn-hands-on/models/01_intro.pth"

model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))

model.eval()

categories = ["Cat(猫)", "Dog(犬)"]

In [ ]:
###
# Cell 03
###

preprocess = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [ ]:
###
# Cell 04
###

csv_path = "/content/drive/MyDrive/cnn-hands-on/data/cats_vs_dogs/labels.csv"
df = pd.read_csv(csv_path)

test_df = df[df["split"] == "test"]

random_row = test_df.sample(n=1).iloc[0]
image_rel_path = random_row["filepath"]
true_class = random_row["class_name"]

random_image_path = os.path.join("/content/drive/MyDrive/cnn-hands-on/data/cats_vs_dogs", image_rel_path)

input_image = Image.open(random_image_path)
plt.imshow(input_image)
plt.title(f"True Label: {true_class}")
plt.axis("off")
plt.show()

input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0)

with torch.no_grad():
    output = model(input_batch)

probabilities = torch.nn.functional.softmax(output[0], dim=0)

for i in range(2):
    score = probabilities[i].item() * 100
    class_name = categories[i]
    print(f"{class_name} の確率: {score:.1f}%")

predicted_idx = torch.argmax(probabilities).item()
predicted_class = categories[predicted_idx]

print(f"判定: {predicted_class}")

In [ ]:
###
# Cell 05
###

print("判定したい画像ファイルを選んでください")
uploaded = files.upload()

if len(uploaded) > 0:
    filename = next(iter(uploaded))

    my_image = Image.open(filename).convert("RGB")
    plt.imshow(my_image)
    plt.axis("off")
    plt.show()

    my_tensor = preprocess(my_image)
    my_batch = my_tensor.unsqueeze(0)

    with torch.no_grad():
        output = model(my_batch)

    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    best_idx = torch.argmax(probabilities).item()
    best_score = probabilities[best_idx].item() * 100
    class_name = categories[best_idx]

    print(f"予想: {class_name} , 確率: {best_score:.1f}%")
else:
    print("画像のアップデートがされませんでした")